# Mission Tasking Service — Quickstart

This notebook is **self-contained**. You do not need to run anything from the README first.

---

## Before you open this notebook (one-time setup)

Do these **3 steps in your terminal**, then come back here:

```bash
# 1. Register the project Python environment as a Jupyter kernel
cd /Users/maks/Desktop/langchain/mission-tasking-service
.venv/bin/python -m ipykernel install --user --name mts --display-name "MTS (Python 3.14)"

# 2. Open this notebook using the project's own Jupyter
.venv/bin/jupyter notebook quickstart.ipynb
```

```
# 3. In the notebook UI: Kernel → Change Kernel → MTS (Python 3.14)
```

Also make sure **Docker Desktop is open and running** before you reach Cell 4.

---

## Then run the cells top to bottom

| Cell | What it does | Manual step? |
|------|-------------|--------------|
| 1 | Paste your API key → writes `.env` | ✏️ Edit the key |
| 2 | Install Python dependencies (`uv sync`) | Auto |
| 3 | Run tests (offline, no Docker needed) | Auto |
| 4 | Start full stack with Docker Compose | Auto (Docker must be open) |
| 5 | Wait for service to be healthy | Auto |
| 6 | Seed the database | Auto |
| 7–8 | Compile missions (calls the LLM) | Auto |
| 9 | Approve a plan | Auto |
| 10 | Verify the plan (simulation) | Auto |
| 11 | Check Prometheus metrics | Auto |
| 12 | Links to Grafana, Prometheus, API docs | Open in browser |
| 13 | Tear everything down | Auto |

## 1. Set your API key

Fill in your key below. This writes `.env` from the example template.

In [ ]:
import re
import shutil

# === LLM provider selection =================================================
# MTS routes to whichever provider you set here. Same API_KEY var works for all;
# the code dispatches via MTS_LLM_PROVIDER + MTS_LLM_MODEL.
#
#   anthropic    → claude-haiku-4-5 | claude-sonnet-4-6 | claude-opus-4-7
#   openai       → gpt-4o-mini | gpt-4o
#   google_genai → gemini-2.5-flash-lite | gemini-2.5-flash | gemini-2.5-pro
# ============================================================================
API_KEY = ""  # <-- paste your key here
MTS_LLM_PROVIDER = "anthropic"  # change if using a different provider
MTS_LLM_MODEL = "claude-haiku-4-5"  # change if using a different model
LANGSMITH_API_KEY = ""  # leave empty unless you have a real LangSmith key

shutil.copy(".env.example", ".env")

with open(".env") as f:
    env = f.read()

# Anchored line replaces — `^API_KEY=` must NOT also match `LANGSMITH_API_KEY=`,
# so we use line-anchored regex instead of plain str.replace.
env = re.sub(r"(?m)^API_KEY=.*$", f"API_KEY={API_KEY}", env)
env = re.sub(r"(?m)^MTS_LLM_PROVIDER=.*$", f"MTS_LLM_PROVIDER={MTS_LLM_PROVIDER}", env)
env = re.sub(r"(?m)^MTS_LLM_MODEL=.*$", f"MTS_LLM_MODEL={MTS_LLM_MODEL}", env)
env = re.sub(r"(?m)^LANGSMITH_API_KEY=.*$", f"LANGSMITH_API_KEY={LANGSMITH_API_KEY}", env)

with open(".env", "w") as f:
    f.write(env)

print(f".env written ✓  (provider={MTS_LLM_PROVIDER}, model={MTS_LLM_MODEL})")

## 2. Install Python dependencies

In [ ]:
!~/.local/bin/uv sync --extra dev

## 3. Run the test suite (offline — no API key needed)

The safety core (physics model + kernel validator) and graph wiring tests all run without a live database or LLM.

In [ ]:
!~/.local/bin/uv run pytest -v

## 4. Start the full stack with Docker Compose

This brings up: MTS service, Postgres/PostGIS, Redis, OTel collector, Prometheus, Grafana.

> **Docker Desktop must be running before this cell.**

In [ ]:
!docker compose -f deploy/docker/docker-compose.yml up --build -d

## 5. Wait for services to be healthy

In [ ]:
import time
import urllib.error
import urllib.request

url = "http://localhost:8000/healthz"
for i in range(30):
    try:
        urllib.request.urlopen(url, timeout=2)
        print(f"MTS is up ✓  (attempt {i + 1})")
        break
    except Exception:
        print(f"waiting... ({i + 1}/30)")
        time.sleep(3)
else:
    print("MTS did not come up in time — check: docker compose logs mts")

## 6. Seed the database (operating areas + drone profiles)

In [ ]:
!docker compose -f deploy/docker/docker-compose.yml exec mts python scripts/seed_db.py

## 7. Compile a mission plan

Send a natural-language command and get back a structured `MissionPlan`.

In [ ]:
import json
import urllib.request

payload = {
    "command": "Patrol the yard perimeter at 60 meters with EO sensor and return to base.",
    "area_id": "yard-simple",
    "operator_clearance": "STANDARD",
    "drone_state": {"drone_profile_id": "long-endurance-quad", "battery_pct": 100.0},
}

req = urllib.request.Request(
    "http://localhost:8000/v1/missions:compile",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req, timeout=120) as resp:
    result = json.loads(resp.read())

print(json.dumps(result, indent=2))

In [ ]:
# Pull out the key fields
plan = result["plan"]
print(f"Status        : {plan['status']}")
print(f"Mission ID    : {plan['mission_id']}")
print(f"Legs          : {len(plan['legs'])}")
print(f"Duration      : {plan['total_duration_s']:.0f}s ({plan['total_duration_s'] / 60:.1f} min)")
print(f"Battery used  : {plan['total_battery_pct']:.1f}%")
print(f"Reserve       : {plan['battery_reserve_pct']:.1f}%")
print(f"Repair loops  : {result['repair_loops']}")
print(f"\nReasoning:\n{plan['reasoning_trace']}")

## 8. Try a mission that should be rejected (NFZ violation)

In [ ]:
payload_bad = {
    "command": "Fly directly over the grain silo.",
    "area_id": "farmland-complex",
    "operator_clearance": "STANDARD",
    "drone_state": {"drone_profile_id": "long-endurance-quad", "battery_pct": 100.0},
}

req2 = urllib.request.Request(
    "http://localhost:8000/v1/missions:compile",
    data=json.dumps(payload_bad).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req2, timeout=120) as resp:
    result_bad = json.loads(resp.read())

plan_bad = result_bad["plan"]
print(f"Status   : {plan_bad['status']}")
# Print the field that matches the status (one or the other, never both populated).
if plan_bad["status"] == "REJECTED":
    print(f"Reasons  : {plan_bad['rejection_reasons']}")
elif plan_bad["status"] == "NEEDS_CLARIFICATION":
    print(f"Questions: {plan_bad['clarification_questions']}")
else:
    print(
        f"Plan     : {len(plan_bad['legs'])} legs, "
        f"{plan_bad['total_duration_s']:.0f}s, "
        f"{plan_bad['total_battery_pct']:.1f}% battery"
    )

## 9. Approve a ready plan

In [ ]:
mission_id = plan["mission_id"]  # from cell 7

if plan["status"] == "READY_FOR_APPROVAL":
    approval = {"mission_id": mission_id, "approve": True, "operator_note": "Looks good, proceed."}
    req3 = urllib.request.Request(
        "http://localhost:8000/v1/missions:approve",
        data=json.dumps(approval).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req3, timeout=30) as resp:
        print(json.dumps(json.loads(resp.read()), indent=2))
else:
    print(f"Plan status is '{plan['status']}' — nothing to approve.")

## 10. Verify the approved plan (execution simulation)

In [ ]:
req4 = urllib.request.Request(
    f"http://localhost:8000/v1/missions/{mission_id}:verify",
    method="POST",
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req4, timeout=30) as resp:
    verify = json.loads(resp.read())

print(json.dumps(verify, indent=2))

## 11. Check Prometheus metrics

In [ ]:
with urllib.request.urlopen("http://localhost:8000/metrics") as resp:
    metrics = resp.read().decode()

# Print only MTS-specific lines
for line in metrics.splitlines():
    if line.startswith("mts_") and not line.startswith("#"):
        print(line)

## 12. Open dashboards

- **Grafana**: http://localhost:3000 — login `admin` / `admin`, import `deploy/grafana-dashboard.json`
- **Prometheus**: http://localhost:9090
- **MTS API docs**: http://localhost:8000/docs

## 13. Tear down

In [ ]:
!docker compose -f deploy/docker/docker-compose.yml down -v